# Loss Function Comparison Experiments

Which loss function is most effective for jaguar re-identification?

**Tested Losses:**
- **Standard Batching Group** (6 losses): ArcFace variants, SubCenter, Cross-Entropy, Focal
- **PK Sampling Group** (3 losses): Triplet (hard/semi-hard), ArcFace+Triplet

**Fixed Settings:**
- Backbone: MegaDescriptor-L-384
- Dataset: JID_Master_Dataset (segmented, deduplicated, closed-set split)
- Epochs: 50

**Evaluation Metrics:**
- Identity-balanced mAP (macro-average across identities)
- CMC@1 (Rank-1 accuracy)
- Closed-set mAP (identities with ≥3 train AND ≥3 test samples)

Results logged to Wandb project: `camera-trap-reidentification`, tags: `loss_comparison`

## Setup

In [1]:
import sys
from pathlib import Path
import logging

# Add src to path
project_root = Path.cwd().parent / "camera-trap-footage"
sys.path.insert(0, str(project_root / "src"))

from jaguars.common.logging_utils import setup_logger
from jaguars.reidentification.config import get_default_config
from jaguars.reidentification.experiments import get_loss_experiments
from jaguars.reidentification.training.train import run_processing as run_training

logger = setup_logger("loss_experiments", level=logging.INFO)
print("✓ Imports successful")

✓ Imports successful


## Configuration

Configure base settings for all loss experiments.

In [2]:
# Get default configuration
config = get_default_config()

# Wandb settings
config.wandb.enabled = True
config.wandb.entity = "jaguars"
config.wandb.project = "camera-trap-reidentification"
config.wandb.tags = ["loss_comparison"]  # Tags for filtering experiments

# Dataset settings
config.dataset.source = "fiftyone"
config.dataset.fo_dataset_name = "JID_Master_Dataset"
config.dataset.fo_split_field = "closed_set_split"
config.dataset.fo_patches_field = "sam3_segmentations"
config.dataset.fo_label_field = "ground_truth"
config.dataset.fo_embeddings_field = "embeddings_BVRA_MegaDescriptor_L_384"

# Backbone settings (fixed)
config.backbone.name = "hf-hub:BVRA/MegaDescriptor-L-384"
config.backbone.pretrained = True
config.backbone.embedding_dim = 1536
config.backbone.input_size = 384

# Training settings
config.training.num_epochs = 50

print(f"✓ Base config loaded")
print(f"  Wandb project: {config.wandb.project}")
print(f"  Wandb tags: {config.wandb.tags}")
print(f"  Dataset: {config.dataset.fo_dataset_name}")
print(f"  Backbone: {config.backbone.name}")
print(f"  Epochs: {config.training.num_epochs}")

✓ Base config loaded
  Wandb project: camera-trap-reidentification
  Wandb tags: ['loss_comparison']
  Dataset: JID_Master_Dataset
  Backbone: hf-hub:BVRA/MegaDescriptor-L-384
  Epochs: 50


## Get Loss Experiments

In [3]:
# Get loss experiments with our custom config
loss_experiments = get_loss_experiments(base_config=config)

print(f"✓ {len(loss_experiments)} loss experiments configured:")
print(f"\nStandard Batching Losses:")
for exp in [e for e in loss_experiments if "standard_batching" in e.base_config.wandb.tags]:
    print(f"  - {exp.name}: {exp.description}")

print(f"\nPK Sampling Losses:")
for exp in [e for e in loss_experiments if "pk_sampling" in e.base_config.wandb.tags]:
    print(f"  - {exp.name}: {exp.description}")

✓ 9 loss experiments configured:

Standard Batching Losses:
  - loss_arcface: ArcFace standard margin (m=0.5, s=64)
  - loss_arcface_soft: ArcFace soft margin (m=0.3, s=64)
  - loss_arcface_hard: ArcFace hard margin (m=0.7, s=64)
  - loss_subcenter_arcface: SubCenter ArcFace (handles intra-class variation)
  - loss_cross_entropy: Cross-Entropy (classification baseline)
  - loss_focal: Focal Loss (handles class imbalance)

PK Sampling Losses:
  - loss_arcface_triplet_pk: ArcFace + Triplet (PK sampling, P=8, K=4)
  - loss_triplet_hard_pk: Triplet with hard mining (PK sampling, P=8, K=4)
  - loss_triplet_semi_hard_pk: Triplet with semi-hard mining (PK sampling, P=8, K=4)


## Run Standard Batching Experiments

ArcFace variants, SubCenter, Cross-Entropy, Focal (random batching)

In [4]:
# Run standard batching experiments
standard_results = {}
standard_experiments = [e for e in loss_experiments if "standard_batching" in e.base_config.wandb.tags]

for experiment in standard_experiments:
    logger.info(f"Running experiment: {experiment.name}")
    logger.info(f"  Description: {experiment.description}")
    logger.info(f"  Tags: {experiment.base_config.wandb.tags}")
    logger.info(f"  Group: {experiment.group}")
    
    try:
        result = run_training(experiment.base_config)
        standard_results[experiment.name] = result
        logger.info(f"✓ {experiment.name} completed")
    except Exception as e:
        logger.error(f"✗ {experiment.name} failed: {e}")
        standard_results[experiment.name] = {"error": str(e)}

print(f"\n✓ All {len(standard_experiments)} standard batching experiments completed")

04:15:43 - jid_logger.loss_experiments - INFO - Running experiment: loss_arcface
04:15:43 - jid_logger.loss_experiments - INFO -   Description: ArcFace standard margin (m=0.5, s=64)
04:15:43 - jid_logger.loss_experiments - INFO -   Tags: ['loss_comparison', 'standard_batching']
04:15:43 - jid_logger.loss_experiments - INFO -   Group: loss_comparison_standard
04:15:43 - jid_logger.reidentification.training - INFO - Starting re-identification training...
04:15:43 - jid_logger.reidentification.training - INFO - Dataset source: fiftyone
04:15:43 - jid_logger.reidentification.training - INFO - Backbone: hf-hub:BVRA/MegaDescriptor-L-384
04:15:43 - jid_logger.reidentification.training - INFO - Device: cuda


04:15:50 - jid_logger.reidentification.training - INFO - Resource validation passed


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /sc/home/philipp.kolbe/.netrc.
wandb: Currently logged in as: hpi-philipp-kolbe to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


04:15:52 - jid_logger.reidentification.training - INFO - Loading dataset...
04:16:06 - jid_logger.reidentification.training - INFO - Dataset loaded:
04:16:06 - jid_logger.reidentification.training - INFO -   Train: 946 samples
04:16:06 - jid_logger.reidentification.training - INFO -   Val: 120 samples
04:16:06 - jid_logger.reidentification.training - INFO -   Num classes: 76
04:16:06 - jid_logger.reidentification.training - INFO - Using pre-computed embeddings
04:16:06 - jid_logger.reidentification.training - INFO - DataLoaders created:
04:16:06 - jid_logger.reidentification.training - INFO -   Train batches: 30
04:16:06 - jid_logger.reidentification.training - INFO -   Val batches: 4
Model initialized:
  Input dim: 1536
  Hidden dim: 512
  Embedding dim: 256
  Num classes: 76
  ArcFace margin: 0.5
  ArcFace scale: 64.0
  Total parameters: 939,264
04:16:06 - jid_logger.reidentification.training - INFO - Loss: arcface
04:16:06 - jid_logger.reidentification.training - INFO - Training com

04:16:06 - jid_logger.reidentification.training - INFO - Train Loss: 39.3772, Train Acc: 0.00%
04:16:06 - jid_logger.reidentification.training - INFO - Val Loss: 37.5329, Val Acc: 0.00%
04:16:06 - jid_logger.reidentification.training - INFO - Val mAP (identity-balanced): 0.2400
04:16:06 - jid_logger.reidentification.training - INFO - Val CMC@1: 0.4466, CMC@5: 0.6990
04:16:06 - jid_logger.reidentification.training - INFO - Learning Rate: 0.000100


...

04:26:19 - jid_logger.reidentification.training - INFO - Train Loss: 5.0312, Train Acc: 49.47%


04:26:19 - jid_logger.reidentification.training - INFO - Val Loss: 19.1497, Val Acc: 28.33%
04:26:19 - jid_logger.reidentification.training - INFO - Val mAP (identity-balanced): 0.5211
04:26:19 - jid_logger.reidentification.training - INFO - Val CMC@1: 0.6505, CMC@5: 0.8058
04:26:19 - jid_logger.reidentification.training - INFO - Learning Rate: 0.000100
04:26:20 - jid_logger.reidentification.training - INFO - 
Epoch 50/50


04:26:20 - jid_logger.reidentification.training - INFO - Train Loss: 4.8894, Train Acc: 52.85%
04:26:20 - jid_logger.reidentification.training - INFO - Val Loss: 19.1547, Val Acc: 30.00%
04:26:20 - jid_logger.reidentification.training - INFO - Val mAP (identity-balanced): 0.5273
04:26:20 - jid_logger.reidentification.training - INFO - Val CMC@1: 0.6505, CMC@5: 0.8252
04:26:20 - jid_logger.reidentification.training - INFO - Learning Rate: 0.000100
04:26:20 - jid_logger.reidentification.training - INFO - Saved best model to data/models/reidentification/best_model.pt
04:26:20 - jid_logger.reidentification.training - INFO - Saved checkpoint to data/models/reidentification/checkpoint_epoch_50.pt
04:26:20 - jid_logger.reidentification.training - INFO - ======================================================================
04:26:20 - jid_logger.reidentification.training - INFO - Training completed!
04:26:20 - jid_logger.reidentification.training - INFO - Best epoch: 50
04:26:20 - jid_logger.r

epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▁▁▁▁▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▇▇▇▇█████
train/batch_acc,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▃▃▂▃▃▃▃▄▄▄▅▄▄▅▅▅▅▆▅▆▇▇███
train/batch_cls_loss,███▆▆▇▇▆▆▅▄▄▅▃▅▄▄▄▄▃▂▄▃▃▂▂▂▁▁▂▁▂▁▁▂▁▁▁▁▁
train/batch_loss,█▇▇▇▆▆▅▅▅▅▄▅▃▃▃▄▃▃▃▃▃▂▃▂▂▃▃▂▂▂▂▂▂▂▁▁▂▁▁▁
train/loss,█▇▇▇▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁
val/acc,▁▁▁▁▂▂▂▃▃▄▄▄▄▅▅▆▆▆▆▆▆▆▆▆▇▇▆▆▇▇▇▇▇▇▇▇▇▇▇█
val/cmc@1,▁▁▂▂▃▃▃▄▄▄▅▅▄▄▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇████▇▇██▇█
val/cmc@10,▁▁▁▂▂▃▃▄▃▄▅▅▅▅▅▅▅▅▅▅▆▅▇▅▆▅▆▆▇█▇▆▇▇███▆█▇
+12,...


04:27:42 - jid_logger.loss_experiments - INFO - ✓ loss_focal completed

✓ All 6 standard batching experiments completed


## Run PK Sampling Experiments

Triplet losses and combined ArcFace+Triplet (PK sampling with P=8, K=4)

In [5]:
# Run PK sampling experiments
pk_results = {}
pk_experiments = [e for e in loss_experiments if "pk_sampling" in e.tags]

for experiment in pk_experiments:
    logger.info(f"Running experiment: {experiment.name}")
    logger.info(f"  Description: {experiment.description}")
    logger.info(f"  Tags: {experiment.tags}")
    logger.info(f"  Group: {experiment.group}")
    
    try:
        result = run_training(experiment.base_config)
        pk_results[experiment.name] = result
        logger.info(f"✓ {experiment.name} completed")
    except Exception as e:
        logger.error(f"✗ {experiment.name} failed: {e}")
        pk_results[experiment.name] = {"error": str(e)}

print(f"\n✓ All {len(pk_experiments)} PK sampling experiments completed")

AttributeError: 'ExperimentConfig' object has no attribute 'tags'

## Summary

Display results from all loss experiments.

In [ ]:
# Print summary of results
import pandas as pd

# Combine results
all_results = {**standard_results, **pk_results}

summary_data = []
for exp_name, result in all_results.items():
    if "error" in result:
        summary_data.append({
            "Experiment": exp_name,
            "mAP": "ERROR",
            "CMC@1": "ERROR",
            "Closed-set mAP": "ERROR",
            "Sampling": "PK" if "pk_sampling" in exp_name else "Standard",
        })
    else:
        # Extract key metrics from comprehensive_metrics
        map_val = result.get("identity_balanced_map", "N/A")
        cmc1 = result.get("cmc@1", "N/A")
        cs_map = result.get("closed_set_map", "N/A")
        
        map_str = f"{map_val:.4f}" if isinstance(map_val, (int, float)) else str(map_val)
        cmc1_str = f"{cmc1:.4f}" if isinstance(cmc1, (int, float)) else str(cmc1)
        cs_str = f"{cs_map:.4f}" if isinstance(cs_map, (int, float)) else str(cs_map)
        
        summary_data.append({
            "Experiment": exp_name,
            "mAP": map_str,
            "CMC@1": cmc1_str,
            "Closed-set mAP": cs_str,
            "Sampling": "PK" if "pk_sampling" in exp_name else "Standard",
        })

summary_df = pd.DataFrame(summary_data)
print("\n=== Loss Comparison Results ===")
print(summary_df.to_string(index=False))
print(f"\nView detailed results at: https://wandb.ai/{config.wandb.entity}/{config.wandb.project}")